# 인공지능과언어 과제 #2 — QLoRA 기반 sLLM Text2SQL 미세조정

In [1]:
# 터미널에서 실행
!nvidia-smi
!python --version

Sat Apr 18 11:05:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
!pip uninstall -y torch torchvision torchaudio bitsandbytes triton transformers peft accelerate datasets huggingface_hub autotrain-advanced -q

In [3]:
!pip install \
    transformers==4.40.1 \
    peft==0.10.0 \
    accelerate==0.29.3 \
    bitsandbytes==0.43.1 \
    triton==2.2.0 \
    datasets==2.19.0 \
    huggingface_hub==0.22.2 \
    autotrain-advanced==0.7.77 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 MB 7.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.4.0 requires huggingface-hub>=0.23.0, but you have huggingface-hub 0.22.2 which is incompatible.
sentence-transformers 5.4.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.1 which is incompatible.


In [6]:
!pip install torch==2.2.1 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 757.2/757.2 MB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 26.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 109.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 82.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 60.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 111.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 853.8 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 13.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 7.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
os.kill(os.getpid(), 9)

In [ ]:
from huggingface_hub import login

HF_TOKEN = "hf_.."
HF_USERNAME = "eunzzang"

login(token=HF_TOKEN)
print("HuggingFace 로그인 완료")

Token has not been saved to git credential helper. Pass `add_to_git_credential=True` if you want to set the git credential as well.
Token is valid (permission: write).
Your token has been saved to /root/.cache/huggingface/token
Login successful
HuggingFace 로그인 완료


In [11]:
import re
import os
import gc
import json
import torch
import pandas as pd
from datasets import load_dataset
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM,BitsAndBytesConfig


def make_prompt(ddl, question, query=''):
    prompt = f"""당신은 SQL을 생성하는 SQL 봇입니다. DDL의 테이블을 활용한 Question을 해결할 수 있는 SQL 쿼리를 생성하세요.

### DDL:
{ddl}

### Question:
{question}

### SQL:
{query}"""
    return prompt


def make_inference_pipeline(model_id):
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4"
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="auto",
        torch_dtype=torch.float16
    )

    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer
    )

    return pipe


def normalize_sql(sql: str) -> str:
    sql = sql.strip().lower()
    sql = re.sub(r'\s+', ' ', sql)
    sql = sql.rstrip(';')
    return sql


def extract_generated_sql(text: str) -> str:
    lines = text.strip().splitlines()
    for line in lines:
        line = line.strip()
        if line and not line.startswith('#'):
            return line
    return text.strip().splitlines()[0] if text.strip() else ""


def exact_match_score(pred, gold):
    return normalize_sql(pred) == normalize_sql(gold)


def evaluate_em(df, pipe, prompt_col='prompt', answer_col='answer',
                batch_size=1):
    prompts = df[prompt_col].tolist()

    outputs = pipe(
        prompts,
        do_sample=False,
        return_full_text=False,
        max_new_tokens=64,
        batch_size=batch_size
    )

    gen_sqls = [extract_generated_sql(o[0]['generated_text']) for o in outputs]

    df = df.copy()
    df['gen_sql'] = gen_sqls
    df['em'] = [
        exact_match_score(pred, gold)
        for pred, gold in zip(gen_sqls, df[answer_col].tolist())
    ]

    return df


print("로컬 환경용 함수 정의 완료")

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


로컬 환경용 함수 정의 완료


In [3]:
import torch, bitsandbytes
print(torch.__version__)
print(torch.cuda.is_available())
print(bitsandbytes.__version__)

2.2.1+cu121
True
0.43.1


In [17]:
# ── 도메인 DDL ───────────────────────────────────────────
DOMAIN_DDL = """
CREATE TABLE products (
    product_id   INT PRIMARY KEY AUTO_INCREMENT,
    name         VARCHAR(100) NOT NULL,
    category     VARCHAR(50),
    price        INT NOT NULL,
    stock        INT DEFAULT 0
);
CREATE TABLE customers (
    customer_id  INT PRIMARY KEY AUTO_INCREMENT,
    name         VARCHAR(50) NOT NULL,
    email        VARCHAR(100) UNIQUE,
    grade        VARCHAR(20) DEFAULT 'NORMAL',
    join_date    DATE
);
CREATE TABLE orders (
    order_id     INT PRIMARY KEY AUTO_INCREMENT,
    customer_id  INT,
    product_id   INT,
    quantity     INT DEFAULT 1,
    order_date   DATETIME,
    status       VARCHAR(20) DEFAULT '주문접수'
);
"""

# ── 21개 질문-SQL 쌍 ─────────────────────────────────────
qa_pairs = [
    # 단순 조회 (5개)
    {"question": "모든 상품의 이름과 가격을 조회해줘",
     "answer":   "SELECT name, price FROM products;"},
    {"question": "재고가 0인 상품 이름을 알려줘",
     "answer":   "SELECT name FROM products WHERE stock = 0;"},
    {"question": "가격이 50000원 이상인 상품을 조회해줘",
     "answer":   "SELECT name, price FROM products WHERE price >= 50000;"},
    {"question": "VIP 등급 고객 목록을 조회해줘",
     "answer":   "SELECT name, email FROM customers WHERE grade = 'VIP';"},
    {"question": "배송완료 상태인 주문 수를 알려줘",
     "answer":   "SELECT COUNT(*) FROM orders WHERE status = '배송완료';"},
    # 정렬 / 집계 (5개)
    {"question": "가격이 가장 비싼 상품 TOP 3를 알려줘",
     "answer":   "SELECT name, price FROM products ORDER BY price DESC LIMIT 3;"},
    {"question": "카테고리별 상품 수를 알려줘",
     "answer":   "SELECT category, COUNT(*) AS cnt FROM products GROUP BY category;"},
    {"question": "카테고리별 평균 가격을 조회해줘",
     "answer":   "SELECT category, AVG(price) AS avg_price FROM products GROUP BY category;"},
    {"question": "2024년에 가입한 고객 수를 알려줘",
     "answer":   "SELECT COUNT(*) FROM customers WHERE YEAR(join_date) = 2024;"},
    {"question": "주문 수량 합계가 가장 많은 상품의 product_id를 알려줘",
     "answer":   "SELECT product_id, SUM(quantity) AS total_qty FROM orders GROUP BY product_id ORDER BY total_qty DESC LIMIT 1;"},
    # JOIN (5개)
    {"question": "주문된 상품 이름과 주문 날짜를 조회해줘",
     "answer":   "SELECT p.name, o.order_date FROM orders o JOIN products p ON o.product_id = p.product_id;"},
    {"question": "고객 이름과 주문한 상품명을 같이 보여줘",
     "answer":   "SELECT c.name AS customer, p.name AS product FROM orders o JOIN customers c ON o.customer_id = c.customer_id JOIN products p ON o.product_id = p.product_id;"},
    {"question": "VIP 고객이 주문한 상품 목록을 조회해줘",
     "answer":   "SELECT DISTINCT p.name FROM orders o JOIN customers c ON o.customer_id = c.customer_id JOIN products p ON o.product_id = p.product_id WHERE c.grade = 'VIP';"},
    {"question": "고객별 총 주문 금액을 알려줘",
     "answer":   "SELECT c.name, SUM(p.price * o.quantity) AS total FROM orders o JOIN customers c ON o.customer_id = c.customer_id JOIN products p ON o.product_id = p.product_id GROUP BY c.customer_id, c.name;"},
    {"question": "한 번도 주문하지 않은 고객 이름을 찾아줘",
     "answer":   "SELECT name FROM customers WHERE customer_id NOT IN (SELECT DISTINCT customer_id FROM orders);"},
    # 서브쿼리 / 복합 (6개)
    {"question": "평균 가격보다 비싼 상품 이름과 가격을 조회해줘",
     "answer":   "SELECT name, price FROM products WHERE price > (SELECT AVG(price) FROM products);"},
    {"question": "주문이 5건 이상인 고객의 customer_id와 주문 수를 알려줘",
     "answer":   "SELECT customer_id, COUNT(*) AS order_cnt FROM orders GROUP BY customer_id HAVING COUNT(*) >= 5;"},
    {"question": "각 카테고리에서 가장 비싼 상품 이름을 조회해줘",
     "answer":   "SELECT category, name, price FROM products WHERE (category, price) IN (SELECT category, MAX(price) FROM products GROUP BY category);"},
    {"question": "이번 달에 주문이 없는 상품 이름을 찾아줘",
     "answer":   "SELECT name FROM products WHERE product_id NOT IN (SELECT DISTINCT product_id FROM orders WHERE DATE_FORMAT(order_date, '%Y-%m') = DATE_FORMAT(NOW(), '%Y-%m'));"},
    {"question": "NORMAL 등급 고객 중 가장 최근에 가입한 고객 이름을 알려줘",
     "answer":   "SELECT name FROM customers WHERE grade = 'NORMAL' ORDER BY join_date DESC LIMIT 1;"},
    {"question": "주문 상태별 주문 수와 총 수량을 조회해줘",
     "answer":   "SELECT status, COUNT(*) AS order_cnt, SUM(quantity) AS total_qty FROM orders GROUP BY status;"},
]

domain_df = pd.DataFrame(qa_pairs)
domain_df['context'] = DOMAIN_DDL

print(f"도메인 데이터: {len(domain_df)}개")
print(domain_df[['question','answer']].to_string())

도메인 데이터: 21개
                                 question                                                                                                                                                                                            answer
0                     모든 상품의 이름과 가격을 조회해줘                                                                                                                                                                 SELECT name, price FROM products;
1                       재고가 0인 상품 이름을 알려줘                                                                                                                                                        SELECT name FROM products WHERE stock = 0;
2                 가격이 50000원 이상인 상품을 조회해줘                                                                                                                                            SELECT name, price FROM products WHERE price >= 50000;
3                      VIP 등급 고객 목록을 조회해줘  

In [18]:
# ko_text2sql train split 로드 (교수님 예제 6.9)
df_sql = load_dataset("shangrilar/ko_text2sql", "origin")["train"]
df_sql = df_sql.to_pandas()
df_sql = df_sql.dropna().sample(frac=1, random_state=42)
df_sql = df_sql.query("db_id != 1")

for idx, row in df_sql.iterrows():
    df_sql.loc[idx, 'text'] = make_prompt(row['context'], row['question'], row['answer'])

for idx, row in domain_df.iterrows():
    domain_df.loc[idx, 'text'] = make_prompt(row['context'], row['question'], row['answer'])

train_df = pd.concat(
    [df_sql[['context','question','answer','text']],
     domain_df[['context','question','answer','text']]],
    ignore_index=True
).sample(frac=1, random_state=42)

os.makedirs('data', exist_ok=True)
os.makedirs('results', exist_ok=True)
train_df.to_csv('data/train.csv', index=False)
print(f"학습 데이터 저장 완료 — ko_text2sql {len(df_sql)}개 + 도메인 {len(domain_df)}개 = 총 {len(train_df)}개")

학습 데이터 저장 완료 — ko_text2sql 33876개 + 도메인 21개 = 총 33897개


In [8]:
BASE_MODEL_ID = "beomi/Yi-Ko-6B"

print("베이스 모델 로드 중...")
hf_pipe = make_inference_pipeline(BASE_MODEL_ID)
print("로드 완료")

# 결과 폴더 생성
os.makedirs("results", exist_ok=True)

# 테스트셋 준비
df_test = load_dataset("shangrilar/ko_text2sql", "origin")["test"]
df_test = df_test.to_pandas()

for idx, row in df_test.iterrows():
    df_test.loc[idx, "prompt"] = make_prompt(row["context"], row["question"])

# 먼저 일부만 테스트 (속도 확인용)
df_test = df_test.head(20)

print("베이스라인 추론 중...")

baseline_df = evaluate_em(
    df_test,
    hf_pipe,
    batch_size=2,
)

baseline_em = baseline_df["em"].mean()
baseline_correct = int(baseline_df["em"].sum())

print(f"\n[베이스라인] Exact Match: {baseline_em:.4f} ({baseline_correct}/{len(baseline_df)})")

baseline_df.to_csv("results/baseline_results.csv", index=False)

# 메모리 해제
del hf_pipe
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("GPU 메모리 해제 완료")

베이스 모델 로드 중...


NameError: name 'make_inference_pipeline' is not defined

In [8]:
!pip install numpy==1.26.4 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 76.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
umap-learn 0.5.12 requires scikit-learn>=1.6, but you have scikit-learn 1.4.2 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires tqdm>=4.67, but you have tqdm 4.66.2 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
hdbscan 0.8.42 requires scikit-learn>=1.6, but you have scikit-learn 1.4.2 which is incompatible.
grain 0.2.16 requires protobuf>=5.28.3, but you have protobuf 4.23.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tensorflow 2.19.0 requires tensorboard~=2.19.

In [13]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [22]:
train_df_small = train_df.sample(n=500, random_state=42)
train_df_small.to_csv("data/train.csv", index=False)

In [24]:
BASE_MODEL     = 'beomi/Yi-Ko-6B'
ADAPTER_DIR    = 'yi-ko-6b-text2sql'      # 로컬 저장 경로
HF_USERNAME = "eunzzang" # HF_USERNAME 정의 추가
HUB_REPO       = f'{HF_USERNAME}/yi-ko-6b-text2sql'

!autotrain llm \
  --train \
  --model {BASE_MODEL} \
  --project-name {ADAPTER_DIR} \
  --data-path data/ \
  --text-column text \
  --lr 2e-4 \
  --batch-size 1 \
  --epochs 1 \
  --block-size 256 \
  --warmup-ratio 0.03 \
  --lora-r 4 \
  --lora-alpha 8 \
  --lora-dropout 0.05 \
  --gradient-accumulation 1 \
  --mixed-precision fp16 \
  --use-peft \
  --quantization int4 \
  --trainer sft

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
INFO     | 2026-04-18 12:28:35 | autotrain.cli.run_llm:run:343 - Running LLM
WARNING  | 2026-04-18 12:28:35 | autotrain.trainers.common:__init__:180 - Parameters supplied but not used: version, backend, inference, func, config, train, deploy
Saving the dataset (1/1 shards): 100% 500/500 [00:00<00:00, 139856.75 examples/s]
Saving the dataset (1/1 shards): 100% 500/500 [00:00<00:00, 159600.61 examples/s]
INFO     | 2026-04-18 12:28:35 | autotrain.backend:create:300 - Starting local training...
INFO     | 2026-04-18 12:28:35 | autotrain.commands:launch_command:327 - ['accelerate', 'launch', '--num_machines', '1', '--num_processes', '1', '--mixed_precision', 'fp16', '-m', 'autot

In [26]:
!ls yi-ko-6b-text2sql

adapter_config.json	   README.md		    tokenizer.json
adapter_model.safetensors  special_tokens_map.json  training_args.bin
autotrain-data		   tokenizer_config.json    training_params.json


In [30]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch

base = "beomi/Yi-Ko-6B"
adapter = "yi-ko-6b-text2sql"

tokenizer = AutoTokenizer.from_pretrained(base)

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    base,
    device_map="auto",
    torch_dtype=torch.float16,
    offload_folder="./offload",
    quantization_config=quant_config # Add quantization_config
)

model = PeftModel.from_pretrained(model, adapter, offload_folder="./offload")

prompt = "사용자 질문: 고객 이름이 김인 주문 조회 SQL 작성"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens=100)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

 사용자 질문: 고객 이름이 김인 주문 조회 SQL 작성:
SELECT * FROM orders WHERE customer_name = '김';


In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch, gc

BASE_MODEL = "beomi/Yi-Ko-6B"
ADAPTER_DIR = "./yi-ko-6b-text2sql"
HUB_REPO = "your-id/yi-ko-6b-text2sql"

print("어댑터 병합 중...")

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="cpu",
    trust_remote_code=True,
    low_cpu_mem_usage=True
)

model = PeftModel.from_pretrained(base, ADAPTER_DIR)
model = model.merge_and_unload()

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Hub 업로드 중: {HUB_REPO}")
model.push_to_hub(HUB_REPO)
tokenizer.push_to_hub(HUB_REPO)

print("완료")

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


어댑터 병합 중...


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

In [1]:
print("미세조정 모델 로드 중...")
hf_pipe_ft = make_inference_pipeline(HUB_REPO)
print("로드 완료")

print("미세조정 모델 추론 중...")
ft_df = evaluate_em(df_test, hf_pipe_ft, max_length=1024, batch_size=4)

ft_em      = ft_df['em'].mean()
ft_correct = int(ft_df['em'].sum())

print(f"\n[베이스라인]  EM: {baseline_em:.4f}  ({baseline_correct}/{len(baseline_df)})")
print(f"[미세조정 후] EM: {ft_em:.4f}  ({ft_correct}/{len(ft_df)})")
print(f"[향상폭]       {ft_em - baseline_em:+.4f}")

ft_df.to_csv('results/finetuned_results.csv', index=False)

미세조정 모델 로드 중...


NameError: name 'make_inference_pipeline' is not defined

In [ ]:
def classify_error(pred, gold):
    p, g = pred.lower(), gold.lower()
    if not pred.strip():
        return "빈 출력 — EOS 조기 종료 또는 프롬프트 포맷 미인식"
    if 'join' in g and 'join' not in p:
        return "JOIN 누락 — 다중 테이블 추론 실패 (학습 데이터 부족)"
    if 'having' in g and 'having' not in p:
        return "HAVING 절 누락 — 집계 후 필터링 패턴 미학습"
    if g.count('select') >= 2 and p.count('select') < 2:
        return "서브쿼리 미생성 — 복합 쿼리 학습 데이터 부족"
    if 'group by' in g and 'group by' not in p:
        return "GROUP BY 누락 — 집계 구조 추론 실패"
    if 'order by' in g and 'order by' not in p:
        return "ORDER BY 누락 — 정렬 조건 미생성"
    gold_tok = set(re.findall(r'\b[a-zA-Z_가-힣]\w*\b', g))
    pred_tok = set(re.findall(r'\b[a-zA-Z_가-힣]\w*\b', p))
    stopwords = {'select','from','where','and','or','as','on','by','in','not',
                 'is','null','count','sum','avg','max','min','distinct','limit'}
    missing = (gold_tok - pred_tok) - stopwords
    if missing:
        return f"컬럼·테이블명 오류 — 누락 토큰: {', '.join(list(missing)[:3])}"
    return "SQL 구조 불일치 — 프롬프트 이해 오류 또는 과소적합"

errors = ft_df[ft_df['em'] == False].copy()
errors['error_cause'] = errors.apply(
    lambda r: classify_error(r['gen_sql'], r['answer']), axis=1
)

print(f"총 오답 케이스: {len(errors)}개\n" + "=" * 65)
for i, (_, row) in enumerate(errors.head(7).iterrows(), 1):
    print(f"[오답 {i}]")
    print(f"  질문      : {row['question']}")
    print(f"  정답 SQL  : {row['answer']}")
    print(f"  예측 SQL  : {row['gen_sql']}")
    print(f"  원인 분석 : {row['error_cause']}")
    print()

errors.to_csv('results/error_analysis.csv', index=False)

cause_counts = errors['error_cause'].value_counts()
print("[오답 원인 분포]")
print(cause_counts.to_string())


In [ ]:
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.family'] = 'NanumGothic'
matplotlib.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# 왼쪽: EM 비교 막대
ax = axes[0]
bars = ax.bar(
    ['베이스라인\n(Yi-Ko-6B)', '미세조정 후\n(QLoRA)'],
    [baseline_em, ft_em],
    color=['#7ea8c4', '#e07b54'],
    width=0.45,
    edgecolor='none'
)
ax.set_ylim(0, 1.0)
ax.set_ylabel('Exact Match Score')
ax.set_title('미세조정 전·후 EM 성능 비교')
for bar, val in zip(bars, [baseline_em, ft_em]):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.02,
            f'{val:.4f}', ha='center', fontsize=12)
ax.axhline(baseline_em, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)

# 오른쪽: 오답 원인 파이
ax2 = axes[1]
top_causes = cause_counts.head(5)
ax2.pie(
    top_causes.values,
    labels=[c[:22] for c in top_causes.index],
    autopct='%1.0f%%',
    colors=['#e07b54','#7ea8c4','#f5c17a','#8abf8a','#c4a0c4'][:len(top_causes)],
    startangle=140,
)
ax2.set_title('오답 원인 분포')

plt.tight_layout()
plt.savefig('results/performance_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("그래프 저장: results/performance_analysis.png")


In [ ]:
print("=" * 60)
print("              실험 결과 요약")
print("=" * 60)
print(f"  베이스라인 EM  : {baseline_em:.4f}  ({baseline_correct}/{len(baseline_df)})")
print(f"  미세조정 후 EM : {ft_em:.4f}  ({ft_correct}/{len(ft_df)})")
print(f"  향상폭         : {ft_em - baseline_em:+.4f}")
print(f"  총 오답 케이스 : {len(errors)}개 (상세 → results/error_analysis.csv)")
print()
print("  하이퍼파라미터")
print(f"    모델       : beomi/Yi-Ko-6B")
print(f"    lora_r     : 16")
print(f"    lora_alpha : 32")
print(f"    quantization: int4")
print(f"    batch_size : 1  (gradient_accumulation=8 → 유효 배치=8)")
print(f"    epochs     : 1,  lr=2e-4")
print("=" * 60)